<a href="https://colab.research.google.com/github/ILikePotatoesALot/Football_Computer_Vision_Match_Analysis/blob/main/FootballModelTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies

!pip install -q roboflow ultralytics

# Download football dataset
from roboflow import Roboflow

# Replace with YOUR API key from Step 1.1
rf = Roboflow(api_key="sWHqlBO2rKnQeWpSffIp")

# Download the exact dataset from the video tutorial
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
dataset = project.version(1).download("yolov11")

print(f"✅ Dataset downloaded to: {dataset.location}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 119.4 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to football-players-detection-1 in yolov11:: 100%|██████████| 1338/1338 [00:00<00:00, 3141.12it/s]

✅ Dataset downloaded to: /content/football-players-detection-1


In [ ]:
# See the image filenames
!ls {dataset.location}/valid/images/ | head -10

08fd33_3_1_png.rf.6f25c835bf6d1828dcf584e5969b1f58.jpg
08fd33_3_3_png.rf.128b8280598b9931fdeeed42b5be4c51.jpg
08fd33_9_8_png.rf.cc61e7ba09940f4606e4464dd621fe2f.jpg
121364_7_9_png.rf.bd5ceb93233525ef03fac0eae292f5ed.jpg
121364_9_2_png.rf.400d952c966048709aa5b421889a4dba.jpg
121364_9_3_png.rf.175a041e339a6a4c918ae7f7141461aa.jpg
2e57b9_3_10_png.rf.f85ed097b3476e84600e0a9409959cf4.jpg
2e57b9_3_8_png.rf.6896c68ab1045f7595468d06c0021b79.jpg
2e57b9_9_1_png.rf.87b260e714b9f6af5288301e51014dc4.jpg
40cd38_7_7_png.rf.734097f181067cdf149fbd63249a51b2.jpg


In [ ]:
# Zip the trained model
!zip -r football_model.zip runs/detect/football_yolo11/weights/

# Download to your computer
from google.colab import files
files.download('football_model.zip')


	zip warning: name not matched: runs/detect/football_yolo11/weights/

zip error: Nothing to do! (try: zip -r football_model.zip . -i runs/detect/football_yolo11/weights/)


FileNotFoundError: Cannot find file: football_model.zip

In [ ]:
# In Colab - find your previous model
!ls -lh runs/detect/*/weights/best.pt


-rw-r--r-- 1 root root 39M Feb  9 09:35 runs/detect/football_optimized/weights/best.pt


In [ ]:
from ultralytics import YOLO

# Load YOLOv11 medium (your current model)
model = YOLO('yolo11m.pt')

# Train on football dataset
print("🔥 Starting training...")
print("⏱️  This will take 30-60 minutes")
print("💡 Tip: Minimize browser tab, it runs in background!\n")

results1 = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=60,              # 50 training cycles
    imgsz=640,              # Image size
    batch=16,               # 16 images at a time
    patience=10,            # Stop if no improvement for 10 epochs
    name='football_yolo11',
    device=0                # Use GPU
)

print("\n✅ Training complete!")
print(f"📂 Model saved at: runs/detect/football_yolo11/weights/best.pt")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🔥 Starting training...
⏱️  This will take 30-60 minutes
💡 Tip: Minimize browser tab, it runs in background!

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/football-players-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0

In [ ]:

from ultralytics import YOLO

# Start fresh from pretrained
model2 = YOLO('yolo11m.pt')

results = model2.train(
    data=f'{dataset.location}/data.yaml',

    # OPTIMIZED FOR SMALL DATASETS
    epochs=80,                # More epochs
    imgsz=640,
    batch=8,                  # SMALLER batch (better for small data)

    # CRITICAL: Lower learning rate
    lr0=0.0005,              # Much lower initial LR
    lrf=0.01,                 # Final LR

    # Prevent overfitting
    weight_decay=0.0005,      # L2 regularization
    warmup_epochs=3,
    warmup_momentum=0.8,

    # More augmentation (creates more training variety)
    degrees=15,               # Rotation
    translate=0.2,            # Translation
    scale=0.5,                # Scaling
    shear=2,                  # Shear
    perspective=0.0001,       # Perspective
    flipud=0.0,               # No vertical flip
    fliplr=0.5,               # Horizontal flip
    mosaic=1.0,               # Mosaic augmentation
    mixup=0.1,                # Mixup augmentation

    # Early stopping
    patience=15,              # Stop if no improvement for 15 epochs

    name='football_optimized',
    device=0
)

print(f"Best mAP: {results.results_dict['metrics/mAP50(B)']:.3f}")


Ultralytics 8.4.13 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/football-players-detection-1/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=football_optimized2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tru

In [ ]:
print("📥 Loading Custom Football Model...")

print("✅ Model loaded!")

# ADD THIS BLOCK:
print("\n🎯 Model Classes:")
print(model.names)
print(f"\nTotal classes: {len(model.names)}")
for class_id, class_name in model.names.items():
    print(f"   {class_id}: {class_name}")
print()


📥 Loading Custom Football Model...
✅ Model loaded!

🎯 Model Classes:
{0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}

Total classes: 4
   0: ball
   1: goalkeeper
   2: player
   3: referee

